In [ ]:
# Snowflake-META Pipeline Reader Example
# This notebook demonstrates how to read control table specs using ControlTableReader
import sys
import os
import json

from snowflake.snowpark import Session
from snowmeta.controltable_reader import ControlTableReader
from snowmeta.snowmeta_pipeline import SnowmetaPipeline

# Configure pipeline reader with control tables and optional filters
pipeline_reader_config = {
    "bronze_control_table": "RAW.SNOWMETA_CONFIG.sample_bronze_control_table",
    "silver_control_table": "RAW.SNOWMETA_CONFIG.sample_silver_control_table",
    "group": "A1"  # Optional: filter by data flow group
}

print("Pipeline Reader Configuration:")
for key, value in pipeline_reader_config.items():
    print(f"  {key}: {value}")

In [ ]:
# Step 1: Get Snowpark Session
# The session is automatically tied to the warehouse assigned to this notebook
session = Session.builder.getOrCreate()
print("Snowpark session established successfully!")
warehouse = session.get_current_warehouse()
print(f"Current warehouse: {warehouse}")



In [ ]:
# Step 2: Initialize ControlTableReader
# Pass the session and pipeline_reader_config
reader = ControlTableReader(
            session=session,
            bronze_control_table="RAW.SNOWMETA_CONFIG.sample_bronze_control_table",
            silver_control_table="RAW.SNOWMETA_CONFIG.sample_silver_control_table"
        )
bronze_specs = reader.get_bronze_control_table()
silver_specs = reader.get_silver_control_table()

print("✅ ControlTable Read successfully!")


In [ ]:
pipeline_bronze_control_table = [
    {
        "source_table": "customer",
        "source_path_dev": "@RAW.ETBANKSFINANCIAL.S3_LANDING_CI/customer/",
        "reader_format": "CSV",
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "CUSTOMER"
    }
]

In [ ]:
pipeline_bronze_control_table = [
    {
        "source_table": "customer",
        "source_path_dev": "@RAW.ETBANKSFINANCIAL.S3_LANDING_CI/customer/",
        "reader_format": "CSV",
        "byos_schema":"@RAW.ETBANKSFINANCIAL.S3_LANDING_CI/myschemafiles/sample_customer_schema.json",
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "CUSTOMER"
    }
]

In [ ]:
pipeline_silver_control_table = [
    {
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "CUSTOMER",
        "silver_database_dev": "ANALYTICS",
        "silver_schema": "FINANCIAL_SILVER",
        "silver_table": "CUSTOMER",
        "silver_cdc_apply_changes": {
         "keys": [
            "customer_id"
         ],
         "sequence_by": "load_timestamp",
         "scd_type": "2",
         "except_column_list": [
            "Op",
            "dmsTimestamp",
            "_rescued_data"
         ]
      }
    }
]

In [ ]:
pipeline = SnowmetaPipeline(session=session)
pipeline.invoke_bronze_pipeline(pipeline_bronze_control_table)

In [ ]:
pipeline.invoke_silver_scd2_pipeline(pipeline_silver_data=pipeline_silver_control_table,   
    pipeline_bronze_data=pipeline_bronze_control_table
)

In [ ]:
pipeline_bronze_control_table = [
    {
        "source_table": "Banks_2022_2023_raw",
        "source_path_dev": "@RAW.ETBANKSFINANCIAL.LANDING/",
        "reader_format": "CSV",
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "Banks_2022_2023"
    },
    {
        "source_table": "Insurance_2022_2023_raw",
        "source_path_dev": "@RAW.ETBANKSFINANCIAL.LANDING/",
        "reader_format": "CSV",
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "Insurance_2022_2023"
    },
    {
        "source_table": "Banks_2023_2024_raw",
        "source_path_dev": "@RAW.ETBANKSFINANCIAL.LANDING/",
        "reader_format": "CSV",
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "Banks_2023_2024"
    },
    {
        "source_table": "Insurance_2023_2024_raw",
        "source_path_dev": "@RAW.ETBANKSFINANCIAL.LANDING/",
        "reader_format": "CSV",
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "Insurance_2023_2024"
    }
]

In [ ]:
pipeline_silver_control_table = [
    {
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "Banks_2022_2023",
        "silver_database_dev": "ANALYTICS",
        "silver_schema": "FINANCIAL_SILVER",
        "silver_table": "Banks_2022_2023",
        "silver_cdc_apply_changes": {
            "keys": ["Banks"],
            "sequence_by": "SN",
            "scd_type": "1"
        }
    },
    {
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "Insurance_2022_2023",
        "silver_database_dev": "ANALYTICS",
        "silver_schema": "FINANCIAL_SILVER",
        "silver_table": "Insurance_2022_2023",
        "silver_cdc_apply_changes": {
            "keys": ["Insurers"],
            "sequence_by": "SN",
            "scd_type": "1"
        }
    },
    {
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "Banks_2023_2024",
        "silver_database_dev": "ANALYTICS",
        "silver_schema": "FINANCIAL_SILVER",
        "silver_table": "Banks_2023_2024",
        "silver_cdc_apply_changes": {
            "keys": ["Banks"],
            "sequence_by": "SN",
            "scd_type": "1"
        }
    },
    {
        "bronze_database_dev": "ANALYTICS",
        "bronze_schema": "FINANCIAL_BRONZE",
        "bronze_table": "Insurance_2023_2024",
        "silver_database_dev": "ANALYTICS",
        "silver_schema": "FINANCIAL_SILVER",
        "silver_table": "Insurance_2023_2024",
        "silver_cdc_apply_changes": {
            "keys": ["Insurers"],
            "sequence_by": "SN",
            "scd_type": "1"
        }
    }
]


In [ ]:
CREATE OR REPLACE FILE FORMAT raw.snowmeta_config.CSV_FILE_FORMAT
  TYPE = CSV
  FIELD_DELIMITER = ','          
  RECORD_DELIMITER = '\n'
  PARSE_HEADER = TRUE                
  FIELD_OPTIONALLY_ENCLOSED_BY = NONE 
  NULL_IF = ('NULL','null','\\N') 
  error_on_column_count_mismatch=false
  COMPRESSION = AUTO; 

CREATE OR REPLACE FILE FORMAT raw.snowmeta_config.JSON_FILE_FORMAT
  TYPE = JSON
  STRIP_OUTER_ARRAY = TRUE
  NULL_IF = ('NULL','null','\\N')
  IGNORE_UTF8_ERRORS = TRUE
  COMPRESSION = AUTO;
  
CREATE OR REPLACE FILE FORMAT raw.snowmeta_config.PARQUET_FILE_FORMAT
  TYPE = PARQUET
  NULL_IF = ('NULL','null','\\N')
  COMPRESSION = AUTO;

CREATE OR REPLACE FILE FORMAT raw.snowmeta_config.XML_FILE_FORMAT
  TYPE = XML
  STRIP_OUTER_ELEMENT = TRUE
  IGNORE_UTF8_ERRORS = TRUE
  PRESERVE_SPACE = FALSE
  DISABLE_AUTO_CONVERT = FALSE
  COMPRESSION = AUTO;
